<a href="https://colab.research.google.com/github/RJFranqui/BodyBuilderBuilder/blob/main/mp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Text embeddings can be used to represent text as numerical vectors.

Embeddings are necessary for various downstream tasks, e.g., search, clustering, translation, and  summarization.

## New tools/technologies

[Apache Beam](https://beam.apache.org/get-started/quickstart/python/) is an open source, unified model for defining both batch and streaming data-parallel processing pipelines.

Apache Beam's [MLTransform](https://beam.apache.org/documentation/transforms/python/elementwise/mltransform/) can apply common machine learning (ML) processing tasks on keyed data, in particular to generate embeddings from text data. Since processing is parallel, unique keys are needed to match output and input.

Hugging Face's [`SentenceTransformers`](https://huggingface.co/sentence-transformers) framework uses Python to generate sentence, text, and image embeddings.


## Install dependencies

Install Apache Beam and the dependencies needed to work with Hugging Face embeddings. The dependencies includes the `sentence-transformers` package, which is required to use the `SentenceTransformerEmbeddings` module. You may need to run this cell twice to get rid of dependency error.


In [ ]:
! pip install apache_beam>=2.53.0 --quiet
! pip install sentence-transformers --quiet
! pip install huggingface_hub --quiet

# Init sentence transformer model

In [ ]:
import tempfile, json, re, gzip
import apache_beam as beam
from apache_beam.ml.transforms.base import MLTransform
from apache_beam.ml.transforms.embeddings.huggingface import SentenceTransformerEmbeddings
# this file is needed for beam to do houskeeping
artifact_location_t5 = tempfile.mkdtemp(prefix='huggingface_')
# we will use sentence-transformers/sentence-t5-large and transform column 'content' of the input data (remaining columns will be unchanged)
embedding = SentenceTransformerEmbeddings(
        model_name='sentence-transformers/sentence-t5-large', columns=['content'])

## To use upload/download you need to login into HGF CLI; use write token created on HGF

In [ ]:
!git config --global credential.helper store
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: write)

## Please make sure your file is on huggingface dataset fdac24/MP3 or upload from your GitHub MP3 fork.

In [ ]:
!huggingface-cli download --repo-type dataset --local-dir . fdac24/MP3 audris.json.gz

audris.json.gz: 100% 9.22k/9.22k [00:00<00:00, 19.1MB/s]
Download complete. Moving file to audris.json.gz
audris.json.gz


# check if the file was uploaded

In [ ]:
!ls -alt

total 36
drwxr-xr-x 1 root root 4096 Nov 11 13:52  .
-rw-r--r-- 1 root root 9224 Nov 11 13:52  audris.json.gz
drwxr-xr-x 3 root root 4096 Nov 11 13:52  .cache
drwxr-xr-x 3 root root 4096 Nov 11 13:49  MP3
-rw-r--r-- 1 root root    0 Nov 11 13:34 '=2.53.0'
drwxr-xr-x 1 root root 4096 Nov 11 13:10  ..
drwxr-xr-x 1 root root 4096 Nov  7 20:56  sample_data
drwxr-xr-x 4 root root 4096 Nov  7 20:56  .config


# Now read in the data

In [ ]:
import json, re, gzip

p = re.compile('^#\s')
def splitSections (text): # READMEs have sections, encode each separately
  res = []
  n = 0
  fline = text.splitlines()[0]
  if not p.match(fline):
    text = "# Preamble\n"+text
  sections = re.split (r'(?m)^#+ (.*)\n', text) # iterate over top-level sections
  nsec = len(sections)
  for i in range(1, len(sections), 2):
    #print (str(i)+ ";" + sections[i+1].strip() + ";" + str(len(sections[i].strip())) + ";" + str(len(sections[i+1].strip())))
    res.append( { 'section': str(n)+": "+ sections[i].strip(), 'content': sections[i+1].strip() } )
    n += 1
  return res

utid="audris"
content = []
fi = gzip.open(f"{utid}.json.gz", 'r')
for line in fi:
   line = line.strip ()
   res = json.loads(line)
   if res['type'] in ('model','data'):  # Run embeddings only for each section of HF READMEs
     for s in splitSections (res['content']):
       content.append({'content':s['content'], 'ID':res['ID'], 'type':res['type'], 'section': s['section']})


# Finally run the pipeline

In [ ]:
fo = gzip.open (f"{utid}_embed.json.gz", 'w') # Create output file
loc=tempfile.mkdtemp () # MLTransform needs temp folder to handle its internal activity
with beam.Pipeline() as pipeline: # with construct runs the pipline at the end, otherwise, do .run() explicitly
  data_pcoll = (
      pipeline
      | "Create Data Source" >> beam.Create(content))  # this is our input
  transformed_pcoll = (
      data_pcoll
      | "Do Embedding" >> MLTransform (write_artifact_location=loc) .with_transform (embedding))  # this defines the transform
  transformed_pcoll | 'Diagnostics' >> beam.Map(lambda x: print(len(x['content'])))   # now can create a graph using pipes and >> as in shell script
  transformed_pcoll | 'Output' >> beam.Map(lambda x: fo.write((json.dumps([ x['ID'],x['type'],x['section'],x['content'] ], ensure_ascii=False)+"\n").encode())) # create json text and encode as bytes; add newline: write is not print
fo.close()  # need to flush the buffer or file will be incomplete



768
768
768
768
768
768
768
768
768
768
768
768
768
768
768


## Check if output is OK

In [ ]:
!zcat audris_embed.json.gz| cut -c1-200|head -1

["GroNLP/T0pp-sharded", "model", "0: Preamble", [-0.0354800783097744, 0.00627742288634181, 0.013050922192633152, 0.02673012763261795, 0.04039746895432472, 0.012702501378953457, -0.012275796383619308, 


# Upload results to HGF
## Make sure the token you have logged in with is a write token

In [ ]:
!huggingface-cli upload --repo-type dataset fdac24/MP4 audris_embed.json.gz

Consider using `hf_transfer` for faster uploads. This solution comes with some limitations. See https://huggingface.co/docs/huggingface_hub/hf_transfer for more details.
audris_embed.json.gz: 100% 108k/108k [00:00<00:00, 361kB/s] 
https://huggingface.co/datasets/fdac24/MP4/blob/main/audris_embed.json.gz
